In [2]:


import os
import pandas as pd
data_folder = '../data/'
path = os.path.join(data_folder)

df = pd.DataFrame()
df2 = pd.DataFrame()
for file in os.listdir(path):
    if 'price_history' in file:
        df2 = pd.read_csv(data_folder + file)
        df = pd.concat([df, df2])
        

meta_path = '../data/all_products_meta.csv'
meta_df = pd.read_csv(meta_path)

df = df.merge(meta_df, on = 'asin')

KEEPA_EPOCH = pd.Timestamp('2011-01-01')
df['listedSince'] = pd.to_datetime(KEEPA_EPOCH + pd.to_timedelta(df['listedSince'], 'min'))
df['datetime'] = pd.to_datetime(df['datetime'])
df['days_since_launch'] = (df['datetime'] - df['listedSince']).dt.days
df = df[df['days_since_launch'] > 0]

df['median'] = df.groupby('asin')['NEW'].transform('median')
df = df[df['NEW'] < 3 * df['median']]
df = df[df['NEW'].notna()]

df = df.set_index('datetime', drop = True)
backup_df = df
df = df.groupby('asin').resample('W')['NEW'].min().reset_index()
df['NEW'] = df.groupby('asin')['NEW'].ffill()

df = df.merge(meta_df[['asin', 'brand', 'model', 'listedSince']], on='asin', how='left')
df['listedSince'] = pd.to_datetime(KEEPA_EPOCH + pd.to_timedelta(df['listedSince'], 'min'))
df['days_since_launch'] = (df['datetime'] - df['listedSince']).dt.days

import plotly.express as px
iphone_13_df = df[df['model'] == 'iPhone 13']
chart = px.line(iphone_13_df, x = 'days_since_launch', y = 'NEW')
chart.show()

In [3]:
all_iphones_df = df[df['brand'] == 'Apple']
print(all_iphones_df.head())

all_iphones_line_chart = px.line(
    all_iphones_df,
    x = 'days_since_launch',
    y = 'NEW',
    color = 'model'
    )

all_iphones_line_chart.show()

         asin   datetime    NEW  brand      model         listedSince  \
0  B07ZPKN6YR 2020-10-18  527.0  Apple  iPhone 11 2019-10-28 07:00:00   
1  B07ZPKN6YR 2020-10-25  527.0  Apple  iPhone 11 2019-10-28 07:00:00   
2  B07ZPKN6YR 2020-11-01  527.0  Apple  iPhone 11 2019-10-28 07:00:00   
3  B07ZPKN6YR 2020-11-08  527.0  Apple  iPhone 11 2019-10-28 07:00:00   
4  B07ZPKN6YR 2020-11-15  527.0  Apple  iPhone 11 2019-10-28 07:00:00   

   days_since_launch  
0                355  
1                362  
2                369  
3                376  
4                383  


In [4]:
all_iphones_df['month'] = all_iphones_df['datetime'].dt.to_period('M')

summary = all_iphones_df.groupby(['asin', 'month'])['NEW'].agg(
    min_price='min',
    max_price='max',
    first_price='first',
    last_price='last'
).reset_index()

# summary.head()

all_iphones_df['first_price'] = all_iphones_df.groupby('model')['NEW'].transform('first')
all_iphones_df['last_price'] = all_iphones_df.groupby('model')['NEW'].transform('last')
all_iphones_df['max_price'] = all_iphones_df.groupby('model')['NEW'].transform('max')
all_iphones_df['min_price'] = all_iphones_df.groupby('model')['NEW'].transform('min')
all_iphones_df['weekly_drop'] = all_iphones_df.groupby('model')['NEW'].diff(1).abs()


summary = all_iphones_df.groupby('model')[['first_price', 'last_price']].first()
summary['total_drop_pct'] = round((summary['first_price'] - summary['last_price']) / summary['first_price'] * 100, 1)
summary = summary.sort_values('total_drop_pct', ascending = False)
summary['max_weekly_drop'] = all_iphones_df.groupby('model')['weekly_drop'].max()
summary.head(7)


,first_price,last_price,total_drop_pct,max_weekly_drop
model,,,,
iPhone 12,779.97,173.00,77.8,103.99
iPhone 11,527.00,159.96,69.6,38.54
iPhone 13,729.99,248.90,65.9,230.01
iPhone 14,799.97,277.95,65.3,562.25
iPhone 15,779.99,378.71,51.4,80.53
iPhone 16,1350.00,670.93,50.3,100.05
iPhone 17,849.00,771.82,9.1,66.97


In [ ]:
#W2 D3 
'''Add a new column `mean_price` to `all_iphones_df` — the mean NEW price for that model,
aligned to every row (same value repeated for all rows of the same model).
Print head() showing model, NEW, and mean_price columns only.'''
all_iphones_df['mean_price'] = all_iphones_df.groupby('model')['NEW'].transform('mean')


'''Build a summary DataFrame with one row per model, columns:
- `mean_price` — mean weekly NEW price
- `std_price` — standard deviation of weekly NEW prices
- `weeks_of_data` — number of weekly price points'''
summary_df = all_iphones_df.groupby('model').agg(mean_price=('NEW', 'mean'), std_price = ('NEW', 'std'), weeks_of_data = ('NEW', 'count')).reset_index()


'''From the full resampled `df` (all ASINs), identify:
- The 2 Samsung ASINs with the most weekly price points
- The 2 Google ASINs with the most weekly price points
'''
top_asins = df.groupby(['brand', 'model', 'asin']).agg(weekly_price_points = ('NEW', 'count')).reset_index()
top_asins = top_asins.sort_values('weekly_price_points', ascending = False)
top_asins.head(12)



'''Filter `df` to: all Apple iPhones + the 4 chosen ASINs.
Compute the decay summary table (first_price, last_price, total_drop_pct) grouped by model.
Print sorted by total_drop_pct descending.'''

filtered_df = df[(df['brand'] == 'Apple') | (df['asin'].isin(['B08L34JQ9C', 'B08VLMQ3KS', 'B08MV7HWFK', 'B09SYSMCX6']))]
filtered_df = df.groupby('model').agg(first_price = ('NEW', 'first'), last_price = ('NEW', 'last')).reset_index()
filtered_df['total_drop_pct'] = round((filtered_df['first_price'] - filtered_df['last_price']) / filtered_df['first_price'] * 100, 1)
filtered_df = filtered_df.sort_values('total_drop_pct', ascending = False)
filtered_df.head()


'''Plot decay curves for all selected models on one px.line chart.
x = days_since_launch, y = NEW, color = brand.
Title: "Price Decay by Brand".
One observation: which brand decays fastest?'''


import plotly.express as px
decay_chart = px.line(x = df['days_since_launch'], y = df['NEW'], color = df['brand'])
# decay_chart.show()


'''Add column `price_pct_of_launch` to the filtered DataFrame:
price as a percentage of that model's first recorded price.
Formula: `NEW / first_price * 100`
You already know how to get first_price per model. Print min, max, and head().
'''

df['first_price'] = df.groupby('model')['NEW'].transform('first')
df['price_pct_of_launch'] = round(df['NEW'] / df['first_price'] * 100, 1)
df.head()

decay_chart_price_pct = px.line(x = df['days_since_launch'], y = df['price_pct_of_launch'], color = df['brand'], title = 'Price decay - pct of launch price by brand')
decay_chart_price_pct.show()

